In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
from enum import Enum

class Strategy(Enum):
    THREESOLDIERS = 1
    ENGULFING = 2
    
class Audit(Enum):
    WEEKLY = 1
    MONTHLY = 2
    
class Interval(Enum):
    MINUTELY_ONEMINUTE = '1m'
    MINUTELY_THREEMINUTE = '3m'
    MINUTELY_FIVEMINUTE = '5m'
    MINUTELY_FIFTEENMINUTE = '15m'
    MINUTELY_THIRTYMINUTE = '30m'

    HOURLY_ONEHOUR = '1h'
    HOURLY_TWOHOUR = '2h'
    HOURLY_FOURHOUR = '4h'
    HOURLY_SIXHOUR = '6h'
    HOURLY_EIGHTHOUR = '8h'
    HOURLY_TWELVEHOUR = '12h'

    DAILY_ONEDAY = '1d'

    WEEKLY_ONEWEEK = '1w'

    MONTHLY_ONEMONTH = '1M'
    
engulfing_body_condition = 2
threesoldiers_body_condition = [1.01,1.02]
#march and nov 21 and march 24 almost same price (all high)
# 3         11           3

In [13]:
file = "../../data/csv/Binance_BTCUSDT_2024_minute_2024_04_02.csv"
# enums
strategy = Strategy.ENGULFING
audit = Audit.MONTHLY
interval = Interval.MINUTELY_ONEMINUTE
strategies = [Strategy.ENGULFING, Strategy.THREESOLDIERS]

usdt_balance = 45000
buy_percentage_init = 10
sell_percentage_init = 13
learning_rate = 1.3
bitcoin_balance: np.longdouble = 0
period = 1 * 24 * 60 # as candles
drop_rows = True
row_upper_limit = 1 * 7 * 24 * 26 # datas upper than
row_lower_limit = 0 # datas lower than
add_each_month = 0 #20 # dollars
period_counter = 1 # add money after n periods

total_balance = usdt_balance + bitcoin_balance
buyPositionsNumber = 0
sellPositionsNumber = 0
buyDuplicateChecker = 0
sellDuplicateChecker = 0
buy_percentage_current = buy_percentage_init
sell_percentage_current = sell_percentage_init
usdt_balance_init = usdt_balance

print(f'chosen strategy: {strategy.name}')

chosen strategy: ENGULFING


In [14]:
# if columns are not set properly, open the file one time and save it under .csv format
# and try again
if drop_rows:
    if row_lower_limit != 0:
        df = pd.read_csv(file,skiprows=row_lower_limit,nrows=(row_upper_limit - row_lower_limit + 1),names = ['Unix' ,'Date' ,'Symbol' ,'Open' ,'High' ,'Low' ,'Close' ,'Volume BTC','Volume USDT','tradecount'])
    else:
        df = pd.read_csv(file,skiprows=row_lower_limit,nrows=(row_upper_limit - row_lower_limit + 1))
       
else:
    df = pd.read_csv(file)

df.head()


,https://www.CryptoDataDownload.com,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9
0,Unix,Date,Symbol,Open,High,Low,Close,Volume BTC,Volume USDT,tradecount
1,1.7121E+12,4/2/2024 23:59,BTCUSDT,65423.46,65464,65423.45,65463.99,16.78874,1098710.109,771
2,1.7121E+12,4/2/2024 23:58,BTCUSDT,65451.6,65451.61,65419,65423.46,8.95503,585988.8676,596
3,1.7121E+12,4/2/2024 23:57,BTCUSDT,65407.62,65451.61,65407.61,65451.61,6.00903,393161.9803,464
4,1.7121E+12,4/2/2024 23:56,BTCUSDT,65420,65431.99,65401.4,65407.61,15.45417,1010926.702,652


In [15]:
df.describe()

,https://www.CryptoDataDownload.com,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9
count,4369,4369,4369,4369,4369,4369,4369,4369,4369,4369
unique,28,4369,2,3989,3552,3615,3981,4368,4369,2031
top,1.71197E+12,Date,BTCUSDT,70500,70500,65680,70426.47,36.09373,Volume USDT,846
freq,167,1,4368,7,13,6,6,2,1,9


In [16]:
df.columns

Index(['https://www.CryptoDataDownload.com', 'Unnamed: 1', 'Unnamed: 2',
       'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7',
       'Unnamed: 8', 'Unnamed: 9'],
      dtype='object')

In [17]:
df.iloc[0]

https://www.CryptoDataDownload.com           Unix
Unnamed: 1                                   Date
Unnamed: 2                                 Symbol
Unnamed: 3                                   Open
Unnamed: 4                                   High
Unnamed: 5                                    Low
Unnamed: 6                                  Close
Unnamed: 7                             Volume BTC
Unnamed: 8                            Volume USDT
Unnamed: 9                             tradecount
Name: 0, dtype: object

In [18]:
if row_lower_limit == 0 and drop_rows:
    df.columns = df.iloc[0]
    df = df.drop(0)
    df.head()

In [19]:
cdf = df[['Date','Open','High','Low','Close']]
cdf.head()

,Date,Open,High,Low,Close
1,4/2/2024 23:59,65423.46,65464,65423.45,65463.99
2,4/2/2024 23:58,65451.6,65451.61,65419,65423.46
3,4/2/2024 23:57,65407.62,65451.61,65407.61,65451.61
4,4/2/2024 23:56,65420,65431.99,65401.4,65407.61
5,4/2/2024 23:55,65448.33,65448.34,65420,65420


In [20]:
cdf.describe()

,Date,Open,High,Low,Close
count,4368,4368,4368,4368,4368
unique,4368,3988,3551,3614,3980
top,4/2/2024 23:59,70500,70500,65680,70426.47
freq,1,7,13,6,6


In [21]:
print(type(cdf['Open'][1]))
print(type(cdf['High'][1]))

<class 'str'>
<class 'str'>


In [22]:
import warnings
warnings.filterwarnings('ignore')

cdf['Open'] = pd.to_numeric(cdf['Open'])
cdf['High'] = pd.to_numeric(cdf['High'])
cdf['Low'] = pd.to_numeric(cdf['Low'])
cdf['Close'] = pd.to_numeric(cdf['Close'])

print(type(cdf['Open'][1]))
print(type(cdf['High'][1]))

<class 'numpy.float64'>
<class 'numpy.float64'>


In [23]:
cdf['isGreen'] = cdf["Close"]
cdf['bodyLen'] = abs(cdf["Close"] - cdf["Open"])
cdf['bodyLen_normalised'] = abs(cdf["Close"] - cdf["Open"]) / cdf['Open']
cdf = cdf.reset_index()
del cdf['index']
cdf.head()

,Date,Open,High,Low,Close,isGreen,bodyLen,bodyLen_normalised
0,4/2/2024 23:59,65423.46,65464.00,65423.45,65463.99,65463.99,40.53,0.000620
1,4/2/2024 23:58,65451.60,65451.61,65419.00,65423.46,65423.46,28.14,0.000430
2,4/2/2024 23:57,65407.62,65451.61,65407.61,65451.61,65451.61,43.99,0.000673
3,4/2/2024 23:56,65420.00,65431.99,65401.40,65407.61,65407.61,12.39,0.000189
4,4/2/2024 23:55,65448.33,65448.34,65420.00,65420.00,65420.00,28.33,0.000433


In [24]:
cdf.describe()

,Open,High,Low,Close,isGreen,bodyLen,bodyLen_normalised
count,4368.000000,4368.000000,4368.000000,4368.000000,4368.000000,4368.000000,4368.000000
mean,68829.769954,68854.260179,68802.078423,68828.811809,68828.811809,33.006369,0.000485
std,1951.562222,1940.069049,1965.657537,1952.190560,1952.190560,38.840566,0.000580
min,64635.180000,64745.960000,64550.000000,64635.190000,64635.190000,0.000000,0.000000
25%,66695.390000,66735.392500,66651.292500,66695.397500,66695.397500,8.395000,0.000121
50%,69634.760000,69655.450000,69617.825000,69634.045000,69634.045000,21.770000,0.000314
75%,70363.937500,70376.000000,70351.182500,70363.937500,70363.937500,43.295000,0.000634
max,71362.000000,71366.000000,71325.470000,71362.000000,71362.000000,713.280000,0.010609


In [25]:
del cdf['High']
del cdf['Low']

for i in range(len(cdf)):
    tmp = cdf.loc[i,cdf.columns.values]
    if tmp['Open'] <= tmp['Close']:
        tmp['isGreen'] = 1
    else:
        tmp['isGreen'] = 0

    cdf.loc[i] = tmp.values
        
cdf.head(10)

,Date,Open,Close,isGreen,bodyLen,bodyLen_normalised
0,4/2/2024 23:59,65423.46,65463.99,1.0,40.53,0.000620
1,4/2/2024 23:58,65451.60,65423.46,0.0,28.14,0.000430
2,4/2/2024 23:57,65407.62,65451.61,1.0,43.99,0.000673
3,4/2/2024 23:56,65420.00,65407.61,0.0,12.39,0.000189
4,4/2/2024 23:55,65448.33,65420.00,0.0,28.33,0.000433
5,4/2/2024 23:54,65454.54,65448.33,0.0,6.21,0.000095
6,4/2/2024 23:53,65417.41,65454.54,1.0,37.13,0.000568
7,4/2/2024 23:52,65480.01,65417.42,0.0,62.59,0.000956
8,4/2/2024 23:51,65504.01,65480.00,0.0,24.01,0.000367
9,4/2/2024 23:50,65488.30,65504.01,1.0,15.71,0.000240


In [26]:
cdf.tail()

,Date,Open,Close,isGreen,bodyLen,bodyLen_normalised
4363,3/30/2024 23:13,69721.79,69705.97,0.0,15.82,0.000227
4364,3/30/2024 23:12,69726.01,69721.79,0.0,4.22,0.000061
4365,3/30/2024 23:11,69740.01,69726.01,0.0,14.00,0.000201
4366,3/30/2024 23:10,69728.76,69740.01,1.0,11.25,0.000161
4367,3/30/2024 23:09,69749.77,69728.76,0.0,21.01,0.000301


In [27]:
def updateTotalBalance(i):
    global usdt_balance, bitcoin_balance, total_balance
    total_balance = usdt_balance + bitcoin_balance * cdf.loc[i]["Close"]

In [28]:
def buyBTC(i):
    global usdt_balance, bitcoin_balance, total_balance,buyPositionsNumber, sellPositionsNumber
    global buyDuplicateChecker,sellDuplicateChecker, learning_rate, buy_percentage_current
    global buy_percentage_init, sell_percentage_current, sell_percentage_init
   
    buyDuplicateChecker += 1
    sellDuplicateChecker = 0
    if buyDuplicateChecker != 3:
        return
    
    buyDuplicateChecker = 0
    buyDuplicateChecker = 0
    print("\033[92m \033[1m{}\033[00m" .format(f"buy opportunity!\n"))        

    if usdt_balance > 0:
        buyPositionsNumber += 1
        
        print(f'current buy percentage: {buy_percentage_current}')
        amountToBuy_usdt = total_balance * buy_percentage_current / 100
        buy_percentage_current *= learning_rate
        sell_percentage_current = sell_percentage_init
        
        if amountToBuy_usdt > usdt_balance:
            amountToBuy_usdt = usdt_balance
            
        addedBitcoinAmount = np.longdouble(amountToBuy_usdt / cdf.loc[i]["Close"])
        usdt_balance -= amountToBuy_usdt
        bitcoin_balance += addedBitcoinAmount
        updateTotalBalance(i)
        print(f'\tbought {amountToBuy_usdt} $ BTC ({addedBitcoinAmount} BTCs) in {cdf.loc[i]["Date"]}')
        print(f"\tnew usdt balance: {usdt_balance}")
        print(f'\tnew bitcoin balance: {bitcoin_balance} ({bitcoin_balance * cdf.loc[i]["Close"]} $)')
        
    else:
        print(f"\tthere isn't any usdt to buy BTC with it!")
        print(f'\tcurrent BTC balance: {bitcoin_balance} ({bitcoin_balance * cdf.loc[i]["Close"]} $)')

In [29]:
def sellBTC(i):
    global usdt_balance, bitcoin_balance, total_balance,buyPositionsNumber, sellPositionsNumber
    global sellDuplicateChecker,buyDuplicateChecker, buy_percentage_current, buy_percentage_init
    global sell_percentage_current, sell_percentage_init, learning_rate
    
    sellDuplicateChecker += 1
    buyDuplicateChecker = 0
    if sellDuplicateChecker != 3:
        return
    
    buyDuplicateChecker = 0
    sellDuplicateChecker = 0
    print("\033[91m \033[1m{}\033[00m" .format(f"sell opportunity!\n"))        
    if bitcoin_balance > 0:
        sellPositionsNumber += 1
        
        currentValueOfAllBitcoins = bitcoin_balance * cdf.loc[i]["Close"]
        
        print(f'current sell percentage: {sell_percentage_current}')
        amountToSell_usdt = total_balance * sell_percentage_current / 100
        sell_percentage_current *= learning_rate
        buy_percentage_current = buy_percentage_init
        
        if amountToSell_usdt > currentValueOfAllBitcoins:
            amountToSell_usdt = currentValueOfAllBitcoins
        
        amountToSell_bitcoin = amountToSell_usdt / cdf.loc[i]["Close"]
        usdt_balance += amountToSell_usdt
        bitcoin_balance -= amountToSell_bitcoin
        updateTotalBalance(i)
        print(f'\tsold {amountToSell_usdt} $ of BTC ({amountToSell_usdt} $) in {cdf.loc[i]["Date"]}')
        print(f"\tnew usdt balance: {usdt_balance}")
        print(f'\tnew bitcoin balance: {bitcoin_balance} ({bitcoin_balance * cdf.loc[i]["Close"]} $)')
        
    else:
        print(f"\tthere isn't any bitcoin to sell!")
        print(f'\tcurrent USDT balance: {usdt_balance} $')

In [43]:
def checkForPosition(i,strategy):
    if strategy == Strategy.THREESOLDIERS:
        if i >= len(cdf) - 2:
            return
        suitableForBuy = False
        suitableForSell = False
    #     print(f'i = {i}:')

        if cdf.loc[0 + i]['isGreen'] == 1.0:
            if cdf.loc[1 + i]['isGreen'] == 1.0:
                if cdf.loc[2 + i]['isGreen'] == 1.0:
                    if cdf.loc[1 + i]['bodyLen'] / cdf.loc[0 + i]['bodyLen'] >= threesoldiers_body_condition[0]:
                        if cdf.loc[2 + i]['bodyLen'] / cdf.loc[1 + i]['bodyLen'] >= threesoldiers_body_condition[1]:
                            suitableForBuy = True
                
        if cdf.loc[0 + i]['isGreen'] == 0.0:
            if cdf.loc[1 + i]['isGreen'] == 0.0:
                if cdf.loc[2 + i]['isGreen'] == 0.0:
                    if cdf.loc[1 + i]['bodyLen'] / cdf.loc[0 + i]['bodyLen'] >= threesoldiers_body_condition[0]:
                        if cdf.loc[2 + i]['bodyLen'] / cdf.loc[1 + i]['bodyLen'] >= threesoldiers_body_condition[1]:
                            suitableForSell = True
                
    if strategy == Strategy.ENGULFING:
        if i >= len(cdf) - 1:
            return
        suitableForBuy = False
        suitableForSell = False
    #     print(f'i = {i}:')

        if cdf.loc[0 + i]['isGreen'] == 1.0:
            if cdf.loc[1 + i]['isGreen'] == 0.0:
#                     if cdf.loc[1 + i]['bodyLen'] / cdf.loc[0 + i]['bodyLen'] >= engulfing_body_condition:
                    if cdf.loc[0 + i]['bodyLen'] / cdf.loc[1 + i]['bodyLen'] >= engulfing_body_condition:
                        suitableForBuy = True
                
        if cdf.loc[0 + i]['isGreen'] == 1.0:
            if cdf.loc[1 + i]['isGreen'] == 0.0:
#                 if cdf.loc[2 + i]['isGreen'] == 0.0: # not sure if this line is neceesary?
#                 if cdf.loc[1 + i]['bodyLen'] / cdf.loc[0 + i]['bodyLen'] >= engulfing_body_condition:
                if cdf.loc[0 + i]['bodyLen'] / cdf.loc[1 + i]['bodyLen'] >= engulfing_body_condition:
                    suitableForSell = True
                

        if suitableForBuy:
    #         print(f'{i} is suitable for buy!')
            buyBTC(i)

        if suitableForSell:
    #         print(f"{i} is suitable for sell!")
            sellBTC(i)

    #     if not (suitableForBuy or suitableForSell):
    #         print(f'{i} is not suitable for buy nor sell!')

In [44]:
def timeout(i):
    global usdt_balance, bitcoin_balance, total_balance,buyPositionsNumber, sellPositionsNumber, counter
    updateTotalBalance(i)
    print("\033[91m \033[1m{}\033[00m" .format(f"\n\t\tCOUNTER REACHER IT'S MAXIMUM!\n"))
    print("\033[91m \033[1m{}\033[00m" .format(f"\t\topened {buyPositionsNumber} buy\
positions and {sellPositionsNumber} sell positions!\n"))
    print("\033[91m \033[1m{}\033[00m" .format(f"\t\tcurrent usdt balance: {usdt_balance}\n"))
    print("\033[91m \033[1m{}\033[00m" .format(f"\t\tcurrent bitcoin balance: {bitcoin_balance} ({bitcoin_balance * cdf.loc[i]['Close']} $)\n"))
    print("\033[91m \033[1m{}\033[00m" .format(f"\t\tcurrent total balance: {total_balance}\n"))
    counter = 0
    buyPositionsNumber = 0
    sellPositionsNumber = 0
    return total_balance

In [45]:
def calculatePeriod():
    global audit, interval, period
    if audit == Audit.MONTHLY:
        if interval == Interval.DAILY_ONEDAY:
            period = 30
        if interval == Interval.HOURLY_FOURHOUR:
            period = 6 * 30
        if interval == Interval.HOURLY_ONEHOUR:
            period = 24 * 30
            
    if audit == Audit.WEEKLY:
        if interval == Interval.DAILY_ONEDAY:
            period = 7
        if interval == Interval.HOURLY_FOURHOUR:
            period = 6 * 7
        if interval == Interval.HOURLY_ONEHOUR:
            period = 24 * 7

In [46]:
totals = []
counter = 0
total_counter = 0

calculatePeriod()
for i in range(len(cdf) -1, -1, -1):
    
    for strategy in strategies:
#         print(f'i = {i}, strategy = {strategy}')
        checkForPosition(i,strategy)
        counter += 1
        if counter == period:
            totals.append(timeout(i))
            total_counter += 1
        if total_counter == period_counter:
            print(f"new month!,\n\t{add_each_month} $ added to USDT balance.")
            usdt_balance += add_each_month
            total_counter = 0

 
		COUNTER REACHER IT'S MAXIMUM!

 		opened 0 buypositions and 0 sell positions!

 		current usdt balance: 45000

 		current bitcoin balance: 0 (0.0 $)

 		current total balance: 45000.0

new month!,
	0 $ added to USDT balance.
 
		COUNTER REACHER IT'S MAXIMUM!

 		opened 0 buypositions and 0 sell positions!

 		current usdt balance: 45000

 		current bitcoin balance: 0 (0.0 $)

 		current total balance: 45000.0

new month!,
	0 $ added to USDT balance.
 
		COUNTER REACHER IT'S MAXIMUM!

 		opened 0 buypositions and 0 sell positions!

 		current usdt balance: 45000

 		current bitcoin balance: 0 (0.0 $)

 		current total balance: 45000.0

new month!,
	0 $ added to USDT balance.
 
		COUNTER REACHER IT'S MAXIMUM!

 		opened 0 buypositions and 0 sell positions!

 		current usdt balance: 45000

 		current bitcoin balance: 0 (0.0 $)

 		current total balance: 45000.0

new month!,
	0 $ added to USDT balance.
 
		COUNTER REACHER IT'S MAXIMUM!

 		opened 0 buypositions and 0 sell positions!

 

In [47]:
print("totals:")
for i in totals:
    print(i)
print(f"min: {min(totals)}, max: {max(totals)}")
# print(f"all buy and no change: {usdt_balance_init * (cdf.iloc[0]['Close'] / cdf.iloc[len(cdf) - 1]['Close'])}")

totals:
45000.0
45000.0
45000.0
45000.0
45000.0
45000.0
min: 45000.0, max: 45000.0


### ############################################################

In [25]:
min(totals)

1395.209797811703